Remove HTML, normalize dimensions (M8x40 to a single format), one search term per product

Provide: Clean product dataframe

# Catalog Cleaning

In [1]:
import pandas as pd
import re

# 1. Read raw catalog data
product_data = pd.read_csv("../data/catalog.csv", dtype=str)
print(f"Loaded {len(product_data):,} products.")
product_data.head(2)

Loaded 74,525 products.


,article_number,product_number,area_code,area_name,group_code,group_name,class_code,class_name,family_code,family_name,model_code,model_name,name,label,description,short_text_usp,keywords,product_label
0,0695941615090 1,0695941615,60,Käsityökalut,6012,Lyönti- ja iskutyökalut,601207,Kirjain- ja numeromeistisarjat,60120703,Numeromeistit,6012070301,Numeromeistit,Numeromeistit,Numeromeistit,NaN,NaN,number punch punch stamps,NUMEROMEISTISARJA 5MM
1,1952004268961 1,1952004268,60,Käsityökalut,6023,Autojen erikoistyökalut,602303,Autojen pakokaasujärjestelmien työkalut,60090102,"Lambda-anturityök., pakok.järj. korjaus",60091001112,Lambda-anturin hylsy,"1/2"" lambda-avain, ura",Lambda-anturin hylsy,"<div class=""marketingText AT_PublicationShortT...","Lambda-, NOX- ja pakokaasuhiukkasantureiden av...",Lambda probe tools,"LAMBDA-AVAIN 1/2"" 24MM"


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 2. Lowercase
    text = text.lower()
    
    # 3. Normalize dimensions: '8 x 40' -> '8x40'
    text = re.sub(r'(\d)\s*x\s*(\d)', r'\1x\2', text)
    
    # 4. Strip 'mm' directly following a dimension: 'm8x40 mm' -> 'm8x40'
    text = re.sub(r'(x\d+(?:[.,]\d+)?)\s*mm\b', r'\1', text)
    
    # 5. Normalize multiple spaces to a single space
    return ' '.join(text.split())

# Quick test
test_cases = ["Kuusiokoloruuvi M8X40 A4", "kuusiokoloruuvi m8 x 40 a4", "kuusiokoloruuvi M8 x 40 mm A4"]
for t in test_cases:
    print(f"'{t}'  -->  '{clean_text(t)}'")

'Kuusiokoloruuvi M8X40 A4'  -->  'kuusiokoloruuvi m8x40 a4'
'kuusiokoloruuvi m8 x 40 a4'  -->  'kuusiokoloruuvi m8x40 a4'
'kuusiokoloruuvi M8 x 40 mm A4'  -->  'kuusiokoloruuvi m8x40 a4'


Lets move to catalog data

In [3]:
# clean title and description
product_data['clean_label'] = product_data['product_label'].apply(clean_text)
product_data['clean_desc'] = product_data['description'].apply(clean_text)

# combine title and description into one unified search text
product_data['search_text'] = (product_data['clean_label'] + ' ' + product_data['clean_desc']).str.strip()

# print first 3 rows to verify
for i in range(3):
    print(f"Product {i}:")
    print("  Label:      ", product_data['clean_label'].iloc[i])
    print("  Search text:", product_data['search_text'].iloc[i][:100], "...\n")

Product 0:
  Label:       numeromeistisarja 5mm
  Search text: numeromeistisarja 5mm ...

Product 1:
  Label:       lambda-avain 1/2" 24mm
  Search text: lambda-avain 1/2" 24mm lambda-, nox- ja pakokaasuhiukkasantureiden avaamiseen ja kiristämiseen kuvau ...

Product 2:
  Label:       teränpidin stgcr 2020 k16
  Search text: teränpidin stgcr 2020 k16 laadukas ja kestävä arno teränpidin ulkosorvaukseen positiivisella kääntöt ...



In [4]:
import os

columns_to_keep = [
    'article_number', 'product_number', 
    'area_name', 'group_name', 'class_name',
    'product_label', 'clean_label', 
    'search_text'
]
clean_catalog = product_data[columns_to_keep].copy()

output_path = "../data/clean/catalog_clean.csv"
clean_catalog.to_csv(output_path, index=False)
print(f"Saved {len(clean_catalog):,} clean products to '{output_path}'.")
print(f"File size: {os.path.getsize(output_path) / (1024*1024):.1f} MB")
clean_catalog.head(3)

Saved 74,525 clean products to '../data/clean/catalog_clean.csv'.
File size: 55.6 MB


,article_number,product_number,area_name,group_name,class_name,product_label,clean_label,search_text
0,0695941615090 1,0695941615,Käsityökalut,Lyönti- ja iskutyökalut,Kirjain- ja numeromeistisarjat,NUMEROMEISTISARJA 5MM,numeromeistisarja 5mm,numeromeistisarja 5mm
1,1952004268961 1,1952004268,Käsityökalut,Autojen erikoistyökalut,Autojen pakokaasujärjestelmien työkalut,"LAMBDA-AVAIN 1/2"" 24MM","lambda-avain 1/2"" 24mm","lambda-avain 1/2"" 24mm lambda-, nox- ja pakoka..."
2,0620119270990 1,0620119270,Materiaalin mekaaninen työstö,Sorvaus,ISO teränpidin ulkosorvaus,TERÄNPIDIN STGCR 2020 K16,teränpidin stgcr 2020 k16,teränpidin stgcr 2020 k16 laadukas ja kestävä ...
